<a href="https://colab.research.google.com/github/epicariello/zone30/blob/main/Zone30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ultimo aggiornamento:** 27 marzo 2026, ore 14:45 UTC  
**Branch:** `claude/urban-comparison-PI9se`  
**Commit:** Unified 5-series comparison plots (incidenti/1000ab, gravità) for BO/OL vs IT/ER/SA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


**Questo notebook è stato utilizzato per la realizzazione della tesi L'effetto dell'istituzione della zona 30 km/h sulla mortalità negli incidenti automobilistici**
  

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import unicodedata

# Carica Comuni Emilia-Romagna
df_comuni_emilia_romagna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Emilia_Romagna.xlsx')
print(f'✅ Emilia-Romagna: {len(df_comuni_emilia_romagna)} comuni')

# Carica Comuni Sardegna
df_comuni_sardegna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Sardegna.xlsx')
print(f'✅ Sardegna: {len(df_comuni_sardegna)} comuni')


In [ ]:
import pandas as pd

# Carica il CSV della popolazione per comune per anno
df_pop = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/popolazione_comuni_per_anno.csv',
    sep=';',
    decimal=',',
    encoding='latin-1'
)
df_pop = df_pop.rename(columns={'codice': 'Codice', 'comune': 'Comune'})
print(f'✅ df_pop pronto: {df_pop.shape[0]:,} comuni')
print(f'Colonne: {df_pop.columns.tolist()[:6]} ...')


In [ ]:
import unicodedata, pandas as pd, numpy as np

def normalizza(s):
    """Rimuove accenti e apostrofi, lowercase — per join robusto sui nomi comuni."""
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", "").replace("`", "")

anni_plot = list(range(2015, 2025))

# ── Caricamento incidenti_urbani ────────────────────────────────────────
df_urban = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/incidenti_urbani.csv',
    sep=';', encoding='utf-8'
)
df_urban = df_urban.rename(columns={'nome_comune': 'Comune'})
df_urban['_key'] = df_urban['Comune'].apply(normalizza)

# ── Join con popolazione ────────────────────────────────────────────────
df_pop_tmp = df_pop.copy()
df_pop_tmp['_key'] = df_pop_tmp['Comune'].apply(normalizza)
pop_cols = ['_key'] + [str(a) for a in anni_plot]
df_m = df_urban.merge(df_pop_tmp[pop_cols], on='_key', how='left')

# ── Maschere gruppi ─────────────────────────────────────────────────────
er_keys   = {normalizza(k) for k in df_comuni_emilia_romagna['Comune'].str.strip()}
sard_keys = {normalizza(k) for k in df_comuni_sardegna['Comune'].str.strip()}

is_bo = df_m['_key'] == normalizza('bologna')
is_ol = df_m['_key'] == normalizza('olbia')
is_er = df_m['_key'].isin(er_keys) & ~is_bo
is_sa = df_m['_key'].isin(sard_keys) & ~is_ol
is_it = ~is_bo & ~is_ol

print(f'incidenti_urbani: {len(df_urban):,} comuni, anni 2010–2024')
print(f'Bologna: {is_bo.sum()} | Olbia: {is_ol.sum()} | ER: {is_er.sum()} | SA: {is_sa.sum()} | IT (excl. BO+OL): {is_it.sum()}')
n_no_pop = df_m[str(anni_plot[0])].isna().sum()
print(f'Comuni senza dato popolazione: {n_no_pop:,} su {len(df_m):,}')


In [ ]:
def inc_per_1000(mask):
    """Incidenti ogni 1.000 abitanti per gruppo, anni 2015–2024."""
    vals = []
    for anno in anni_plot:
        inc = pd.to_numeric(df_m.loc[mask, f'{anno}_incidenti'], errors='coerce').sum()
        pop = pd.to_numeric(df_m.loc[mask, str(anno)], errors='coerce').sum()
        vals.append(inc / pop * 1000 if pop > 0 else float('nan'))
    return vals

def gravita(mask):
    """(Morti+Feriti)/Incidenti per gruppo, anni 2015–2024."""
    vals = []
    for anno in anni_plot:
        morti  = pd.to_numeric(df_m.loc[mask, f'{anno}_morti'],     errors='coerce').sum()
        feriti = pd.to_numeric(df_m.loc[mask, f'{anno}_feriti'],    errors='coerce').sum()
        inc    = pd.to_numeric(df_m.loc[mask, f'{anno}_incidenti'], errors='coerce').sum()
        vals.append((morti + feriti) / inc if inc > 0 else float('nan'))
    return vals

s_bo_inc,  s_bo_grav  = inc_per_1000(is_bo), gravita(is_bo)
s_ol_inc,  s_ol_grav  = inc_per_1000(is_ol), gravita(is_ol)
s_it_inc,  s_it_grav  = inc_per_1000(is_it), gravita(is_it)
s_er_inc,  s_er_grav  = inc_per_1000(is_er), gravita(is_er)
s_sa_inc,  s_sa_grav  = inc_per_1000(is_sa), gravita(is_sa)

print('✅ Serie temporali calcolate (2015–2024):')
print(f'  Bologna  inc/1000ab: {[round(v,3) for v in s_bo_inc]}')
print(f'  Olbia    inc/1000ab: {[round(v,3) for v in s_ol_inc]}')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

C_BO = '#2166ac'   # blu — Bologna / Emilia-Romagna
C_OL = '#d6604d'   # rosso-arancio — Olbia / Sardegna
C_IT = '#888888'   # grigio — Italia

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(anni_plot, s_it_inc, color=C_IT, lw=1.5,  ls='-',  label='Italia (excl. BO, OL)')
ax.plot(anni_plot, s_er_inc, color=C_BO, lw=1.8,  ls='--', label='Emilia-Romagna (excl. BO)')
ax.plot(anni_plot, s_sa_inc, color=C_OL, lw=1.8,  ls='--', label='Sardegna (excl. OL)')
ax.plot(anni_plot, s_bo_inc, color=C_BO, lw=2.5,  ls='-',  label='Bologna')
ax.plot(anni_plot, s_ol_inc, color=C_OL, lw=2.5,  ls='-',  label='Olbia')

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (gen 2022)')

ax.set_title('Incidenti stradali urbani ogni 1.000 abitanti (2015\u20132024)', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('Incidenti ogni 1.000 ab.')
ax.set_xticks(anni_plot)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(anni_plot, s_it_grav, color=C_IT, lw=1.5,  ls='-',  label='Italia (excl. BO, OL)')
ax.plot(anni_plot, s_er_grav, color=C_BO, lw=1.8,  ls='--', label='Emilia-Romagna (excl. BO)')
ax.plot(anni_plot, s_sa_grav, color=C_OL, lw=1.8,  ls='--', label='Sardegna (excl. OL)')
ax.plot(anni_plot, s_bo_grav, color=C_BO, lw=2.5,  ls='-',  label='Bologna')
ax.plot(anni_plot, s_ol_grav, color=C_OL, lw=2.5,  ls='-',  label='Olbia')

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (gen 2022)')

ax.set_title('Gravit\u00e0 incidenti urbani: (Morti+Feriti)/Incidenti (2015\u20132024)', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('(Morti+Feriti) / Incidenti')
ax.set_xticks(anni_plot)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
